# Exploring 2026 hitter approach stats

A light exploratory pass over the per-hitter table built by `pull_hitter_data.py`.
Goal: get a feel for **distributions**, **outliers**, and **correlations** among the five
approach stats before clustering.

The five stats (all on a 0–100 scale):
- `k_pct` — strikeout %
- `bb_pct` — walk %
- `chase_pct` — chase % (swings on out-of-zone pitches)
- `whiff_pct` — whiff % (misses per swing)
- `barrel_pct` — barrel % (barrels per batted-ball event, from Savant)

We restrict to **PA ≥ 150** to drop low-sample pitchers/bench players that would distort the picture.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

MIN_PA = 150
STATS = ["k_pct", "bb_pct", "chase_pct", "whiff_pct", "barrel_pct"]
LABELS = {
    "k_pct": "Strikeout %",
    "bb_pct": "Walk %",
    "chase_pct": "Chase %",
    "whiff_pct": "Whiff %",
    "barrel_pct": "Barrel %",
}

raw = pd.read_csv("hitter_stats_2026.csv")
df = raw[raw["pa"] >= MIN_PA].copy().reset_index(drop=True)
print(f"{len(raw)} hitters total -> {len(df)} with PA >= {MIN_PA}")
df.head()

## Summary statistics

Quick numeric overview — center, spread, and range of each stat.

In [ ]:
df[["pa"] + STATS].describe().round(2)

## Distributions

Histogram + KDE for each stat. We're looking for shape: is it roughly normal, skewed, or
**bimodal**? Bimodal stats hint that hitters naturally separate into groups — a good sign for clustering.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, stat in zip(axes, STATS):
    sns.histplot(df[stat], kde=True, ax=ax, color="steelblue")
    ax.set_title(LABELS[stat])
    ax.set_xlabel("")
axes[-1].axis("off")  # 6th panel unused (5 stats)
fig.suptitle(f"Distributions of approach stats (PA >= {MIN_PA}, n={len(df)})", y=1.02)
plt.tight_layout()
plt.show()

## Outliers

Boxplots flag extreme values; the table below ties those extremes to actual players
(top-5 and bottom-5 for each stat).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df[STATS], ax=ax)
ax.set_xticklabels([LABELS[s] for s in STATS])
ax.set_title(f"Approach-stat spread & outliers (PA >= {MIN_PA})")
plt.tight_layout()
plt.show()

In [ ]:
def extremes(stat, n=5):
    """Top-n and bottom-n players for a stat."""
    s = df.sort_values(stat, ascending=False)
    top = s.head(n)[["name", stat]].to_string(index=False)
    bot = s.tail(n)[["name", stat]].to_string(index=False)
    print(f"=== {LABELS[stat]} ===")
    print("Highest:\n" + top)
    print("Lowest:\n" + bot + "\n")

for stat in STATS:
    extremes(stat)

## Relationships & correlations

How do the five stats move together? Strong correlations tell us which stats carry redundant
information (and shape how clusters will form).

In [ ]:
corr = df[STATS].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
    square=True, xticklabels=[LABELS[s] for s in STATS],
    yticklabels=[LABELS[s] for s in STATS], ax=ax,
)
ax.set_title("Correlation between approach stats")
plt.tight_layout()
plt.show()

In [ ]:
import itertools

# Only the 10 unique stat pairs — no self-vs-self diagonal, no mirrored duplicates.
pairs = list(itertools.combinations(STATS, 2))
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (x, y) in zip(axes.ravel(), pairs):
    sns.scatterplot(x=df[x], y=df[y], ax=ax, alpha=0.5, s=25, color="steelblue")
    ax.set_xlabel(LABELS[x])
    ax.set_ylabel(LABELS[y])
fig.suptitle("Pairwise relationships between approach stats", y=1.02)
plt.tight_layout()
plt.show()

## Takeaways

From the PA ≥ 150 set (n = 285):
- **A clear power axis:** K% ↔ whiff% are tightly linked (r ≈ 0.88), and both correlate with barrel% (K%–barrel% ≈ 0.52, whiff%–barrel% ≈ 0.58). This is the "swing hard, accept the miss, do damage" profile — these three largely move as one direction.
- **A separate discipline axis:** walk% and chase% are strongly *negatively* correlated (r ≈ -0.73) — patient hitters don't expand the zone. Chase% is essentially uncorrelated with K%/whiff%/barrel% (≈ 0.0–0.1), so plate discipline is a distinct dimension from raw power, not a proxy for it.
- **Implication for clustering:** the five stats really carry ~2 underlying signals (power & discipline), so standardizing features before K-means/DBSCAN/GMM matters, and clusters should separate more along these two axes than along any single stat. Distributions look largely unimodal (no obvious natural gap), so cluster *count* will be a modeling choice to validate, not something the data hands us for free.